# Pattern 3: Orchestrator and subagents

**Idea:** a central "orchestrator" model call looks at the task and **decides at runtime** how to split it into subtasks. Each subtask goes to a "subagent," a separate call with its **own fresh context**. A final call synthesizes the results.

```
                         ┌─▶ [subagent A] ─┐
task ─▶ [orchestrator] ──┼─▶ [subagent B] ─┼─▶ [synthesizer] ─▶ answer
        (plans subtasks) └─▶ [subagent C] ─┘
```

**How is this different from parallelization (notebook 02)?** In notebook 02 *you* hard-coded three reviewers. Here the *model* decides how many subtasks there are and what they are. Ask about a different topic and you get a different plan.

**Key mechanic, context isolation:** subagents do **not** share memory with the orchestrator or each other. Each one only sees what the orchestrator explicitly passes in. If the orchestrator forgets to pass something, the subagent simply doesn't know it. This is the most important thing to understand about multi-agent designs.

**Why it exists:** big, open-ended tasks where you can't predict the sub-steps in advance. Isolation also keeps each subagent's context small and focused.

**Trade-off:** more calls, harder to predict, and errors in the plan propagate. Only use it when fixed workflows aren't enough.

**Java analogy:** a dispatcher service that reads a request, builds a list of jobs at runtime, sends each to a stateless worker (think a stateless `@Service` or a Lambda), then aggregates. Stateless workers only know what's in the message you send them. Same with subagents.

## Step 0: Setup (same in every notebook)

Every notebook starts with the same setup cells, so you can open any of them alone without running the others first.

1. **Load configuration** from the `.env` file in the repo root. The variable names are the ones the Anthropic SDK and Claude Code already use (`ANTHROPIC_BASE_URL`, `ANTHROPIC_API_KEY` or `ANTHROPIC_AUTH_TOKEN`, `ANTHROPIC_MODEL`), so the same `.env` can serve both.
2. **Create the client** with the official `anthropic` Python SDK. It sends requests to `ANTHROPIC_BASE_URL`: Anthropic's own API, or any gateway that exposes the Anthropic Messages API.
3. **Define a helper**, `call_claude()`, that wraps one API call and prints everything: the prompt going in, the text coming out, token usage, and timing.

> **Java analogy:** `.env` + `python-dotenv` is like `application.properties` + Spring's `@Value`. The client is a `RestTemplate`/`WebClient` bean configured with a base URL. Swapping the endpoint is a config change, not a code change, just like pointing a JDBC URL at a different database that speaks the same protocol.

In [ ]:
# --- Cell 0.1: load configuration from .env ---------------------------------
import os
import json
import time
from dotenv import load_dotenv, find_dotenv

# find_dotenv(usecwd=True) walks UP from the notebook's folder until it finds a .env file.
# That's why the .env can live in the repo root while notebooks live in notebooks/.
env_path = find_dotenv(usecwd=True)
print(f"[setup] .env file found at: {env_path or 'NOT FOUND (using existing environment variables)'}")
# override=False: variables already set in your shell win over the file.
load_dotenv(env_path, override=False)

BASE_URL = os.getenv("ANTHROPIC_BASE_URL", "https://api.anthropic.com")
MODEL = os.getenv("ANTHROPIC_MODEL")
# Direct Anthropic access uses ANTHROPIC_API_KEY (sent as x-api-key).
# Gateways usually expect ANTHROPIC_AUTH_TOKEN (sent as Authorization: Bearer ...).
API_KEY = os.getenv("ANTHROPIC_API_KEY")
AUTH_TOKEN = os.getenv("ANTHROPIC_AUTH_TOKEN")

# Fail fast with a clear message instead of a confusing HTTP 401 later.
assert API_KEY or AUTH_TOKEN, "Set ANTHROPIC_API_KEY or ANTHROPIC_AUTH_TOKEN in .env (see .env.example)."
assert MODEL, "ANTHROPIC_MODEL is missing. Example: claude-sonnet-4-5"

def mask(secret):
    """Never print a full secret. Show just enough to confirm which one is loaded."""
    return None if not secret else secret[:4] + "..." + secret[-4:]

print(f"[setup] ANTHROPIC_BASE_URL   = {BASE_URL}")
print(f"[setup] ANTHROPIC_API_KEY    = {mask(API_KEY)}")
print(f"[setup] ANTHROPIC_AUTH_TOKEN = {mask(AUTH_TOKEN)}")
print(f"[setup] ANTHROPIC_MODEL      = {MODEL}")

In [ ]:
# --- Cell 0.2: create the Anthropic client -----------------------------------
import anthropic

# Optional: extra HTTP headers, one "Name: value" per line.
# Leave this unset when calling Anthropic directly. Set it only if your gateway
# needs headers of its own (for example, routing or tenant identification).
# Same variable name and format that Claude Code uses.
extra_headers = {}
for line in os.getenv("ANTHROPIC_CUSTOM_HEADERS", "").splitlines():
    if ":" in line:
        name, value = line.split(":", 1)
        extra_headers[name.strip()] = value.strip()
print(f"[setup] custom headers: {[f'{k}: {mask(v)}' for k, v in extra_headers.items()] or 'none'}")

client = anthropic.Anthropic(
    api_key=API_KEY,
    auth_token=AUTH_TOKEN,
    base_url=BASE_URL,
    default_headers=extra_headers,
)
print(f"[setup] Anthropic SDK version: {anthropic.__version__}")
print(f"[setup] Client ready. Every request goes to {BASE_URL}/v1/messages")

In [ ]:
# --- Cell 0.3: a logging helper around ONE model call -------------------------
# Everything in these notebooks is built from this single building block.
# The patterns are not new API features: they are just different ways of
# arranging plain calls like this one with ordinary Python code.

def call_claude(prompt, system=None, label="call", max_tokens=1024, verbose=True):
    """Send one user message to Claude and return the response text.

    prompt     -> the user message (str)
    system     -> optional system prompt: the model's role and rules
    label      -> a name for this call so the logs are easy to follow
    verbose    -> print the full prompt and response (set False inside threads)
    """
    if verbose:
        print("=" * 78)
        print(f"[{label}] >>> REQUEST")
        if system:
            print(f"[{label}] system: {system}")
        print(f"[{label}] user:\n{prompt}")

    started = time.perf_counter()
    kwargs = {"model": MODEL, "max_tokens": max_tokens,
              "messages": [{"role": "user", "content": prompt}]}
    if system:
        kwargs["system"] = system
    response = client.messages.create(**kwargs)
    elapsed = time.perf_counter() - started

    # response.content is a LIST of blocks. For plain text calls there is one text block.
    text = "".join(block.text for block in response.content if block.type == "text")

    if verbose:
        print(f"[{label}] <<< RESPONSE ({elapsed:.2f}s, "
              f"in={response.usage.input_tokens} tok, out={response.usage.output_tokens} tok, "
              f"stop_reason={response.stop_reason})")
        print(text)
        print("=" * 78)
    if response.stop_reason == "max_tokens":
        print(f"[{label}] WARNING: response was cut off by max_tokens")
    return text


def parse_json(text):
    """Models sometimes wrap JSON in ```json fences. Strip them, then parse."""
    cleaned = text.strip()
    if cleaned.startswith("```"):
        cleaned = cleaned.split("\n", 1)[1].rsplit("```", 1)[0]
    return json.loads(cleaned)

print("[setup] call_claude() and parse_json() are defined")

In [ ]:
# --- Cell 0.4: smoke test. If this works, your .env and gateway are fine -------
call_claude("Reply with exactly: gateway OK", label="smoke-test", max_tokens=20)

## Step 1: The task

An open-ended question where the right sub-questions aren't obvious in advance.

In [ ]:
task = ("Our team is deciding whether to migrate a Spring Boot monolith to microservices. "
        "Give us a short decision brief.")
print(f"[task] {task}")

## Step 2: Orchestrator plans the subtasks

The orchestrator returns JSON so our code can loop over the plan. Note: **it does not do the work**, it only plans.

In [ ]:
orchestrator_system = (
    "You are an orchestrator. Break the user's task into 2 to 4 independent subtasks "
    "for specialist subagents. Respond ONLY with a JSON array of objects with keys "
    "'role' (a short specialist title) and 'instruction' (what that specialist must produce, "
    "including all context they need, because they cannot see the original task)."
)

plan_raw = call_claude(task, system=orchestrator_system, label="orchestrator")
plan = parse_json(plan_raw)

print(f"[orchestrator] planned {len(plan)} subtasks (decided by the model, not by us):")
for i, sub in enumerate(plan, 1):
    print(f"   {i}. {sub['role']}: {sub['instruction']}")

## Step 3: Run each subagent with an isolated context

Look closely at the request each subagent sends: it contains **only its own instruction**. It never sees the original task text or the other subagents' work. That's why the orchestrator's prompt says "including all context they need."

We run them one by one here so the logs read top to bottom. They're independent, so in production you'd run them in parallel like in notebook 02.

In [ ]:
subagent_results = []

for i, sub in enumerate(plan, 1):
    subagent_system = f"You are a {sub['role']}. Be concise: max 100 words."
    # The ONLY context this subagent gets is the instruction string below.
    result = call_claude(sub["instruction"], system=subagent_system, label=f"subagent-{i}")
    subagent_results.append({"role": sub["role"], "result": result})

print(f"[subagents] collected {len(subagent_results)} results")

## Step 4: Synthesize

The synthesizer gets the original task **plus** every subagent result. Again, the only way information reaches it is by us putting it in the prompt.

In [ ]:
synth_input = f"Original task: {task}\n\nSpecialist findings:\n"
for r in subagent_results:
    synth_input += f"\n### {r['role']}\n{r['result']}\n"

print(f"[synthesizer] input size: {len(synth_input)} characters from {len(subagent_results)} subagents")

synthesizer_system = ("You are an editor. Combine the specialist findings into one decision brief "
                      "with a clear recommendation. Max 200 words.")
final_brief = call_claude(synth_input, system=synthesizer_system, label="synthesizer")

## Step 5: See context isolation fail on purpose

Here we send a subagent a vague instruction that relies on context it doesn't have. It has no idea what "the migration" is. This is the classic multi-agent bug: the coordinator *assumes* the worker knows something.

In [ ]:
call_claude("Evaluate the risks of the migration we discussed.",
            system="You are a risk analyst. Be concise.", label="isolated-subagent")
print("[lesson] The subagent never saw the original task. Context must be passed explicitly.")

## Recap

| Role | Sees | Produces |
|---|---|---|
| Orchestrator | the task | a plan (JSON) |
| Subagent | **only** its instruction | one focused result |
| Synthesizer | task + all results | the final answer |

The model now decides *what* the subtasks are, but your Python code still decides *that* there is a plan step, a worker step, and a synthesis step. Full model control over the loop is the agent pattern (notebook 05).

## Experiments to try

1. Change `task` to something unrelated (for example, planning a team offsite) and watch the plan change shape.
2. Remove "including all context they need" from `orchestrator_system`. Do the subagent answers get vaguer?
3. Print `len(sub["instruction"])` for each subtask. Short instructions are a warning sign of missing context.